In [0]:
# Databricks notebook source
# =============================================================================
# SILVER LAYER : Warehouse Transformation (UNITY CATALOG PROD STANDARD)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

# ──────────────────────────────────────────────────────────────────────────────
# 1. ENVIRONMENT CONFIGURATION & OPTIMIZATIONS
# ──────────────────────────────────────────────────────────────────────────────
# Kill the small file problem automatically!
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_warehouse")
print("⚪ INITIALIZING PROD SILVER PIPELINE : WAREHOUSE ENTITY")

CATALOG_NAME = "prx"
BRONZE_SCHEMA = "prx_bronze"
SILVER_SCHEMA = "prx_silver"

# Fully Qualified Unity Catalog Table Names
BRONZE_TABLE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_warehouse"
SILVER_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.warehouse"
EXCEPTION_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.central_exceptions"

# External Paths for initial creation
STORAGE_ACCOUNT = "stpraxaslakehouse"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/warehouse"
EXCEPTION_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/central_exceptions"

MERGE_KEYS = ["WHSrcId"]

# Ensure Silver Schema Exists Securely
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Capture true executing user for Auditing
executing_user = spark.sql("SELECT current_user()").collect()[0][0]

# ──────────────────────────────────────────────────────────────────────────────
# 2. UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def safe_str(c):
    return F.coalesce(F.col(c).cast(StringType()), F.lit(""))

def safe_int(c):
    return F.coalesce(F.col(c).cast(IntegerType()), F.lit(0))

def empty_str_to_null_ts(c):
    return F.when(
        F.col(c).isNull() | (F.trim(F.col(c)) == ""), None
    ).otherwise(F.col(c)).cast(TimestampType())

# ──────────────────────────────────────────────────────────────────────────────
# 3. READ & TRANSFORMATIONS
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Reading Bronze Warehouse Data from Unity Catalog...")
bronze_df = spark.table(BRONZE_TABLE)

logger.info("Applying Business Transformations...")
silver_df = bronze_df.select(
    # PRIMARY KEYS & Foreign Keys
    safe_str("ID").alias("WHSrcId"),
    safe_str("Division").alias("DivSrcId"),
    safe_str("DefaultStorageLocation").alias("WHDefStorLocSrcId"),
    safe_str("ManagerUser").alias("ContactMngrSrcId"),

    # Warehouse Details
    safe_str("Code").alias("WHKey"),
    safe_str("Division").alias("DivKey"),
    safe_str("Description").alias("WHName"),
    safe_str("Description").alias("WHDesc"),
    safe_str("DefaultStorageLocationCode").alias("WHDefStorLocKey"),
    safe_str("DefaultStorageLocationDescription").alias("WHDefStorLocDesc"),

    # Status Logic
    F.when(F.col("UseStorageLocations") == 1, F.lit("Active")).otherwise(F.lit("InActive")).alias("WHStatus"),

    # Empty Address Fields (Maintained from legacy script)
    F.lit("").alias("AddrKey"),
    F.lit("").alias("AddrLn1"),
    F.lit("").alias("AddrLn2"),
    F.lit("").alias("AddrLn3"),
    F.lit("").alias("City"),
    F.lit("").alias("State"),
    F.lit("").alias("Country"),
    F.lit("").alias("ZipCd"),
    F.lit("").alias("TelPhNo"),
    F.lit("").alias("FaxNo"),

    # Communication & Contact
    safe_str("EMail").alias("EmailAddr"),
    F.lit("").alias("ContactKey"),
    F.lit("").alias("ContactName"),

    # Flags
    safe_int("Main").alias("IsMainFl"),
    F.lit(0).alias("IsSalesFl"),

    # Source System Audit Dates & Lineage
    empty_str_to_null_ts("Created").alias("CreatedDt"),
    safe_str("Creator").alias("CreatedById"),
    safe_str("CreatorFullName").alias("CreatedBy"),
    empty_str_to_null_ts("Modified").alias("UpdatedDt"),
    safe_str("Modifier").alias("UpdatedById"),
    safe_str("ModifierFullName").alias("UpdatedBy"),

    # Enterprise Databricks Pipeline Audit
    F.current_timestamp().alias("SysCreatedDt"),
    F.current_timestamp().alias("SysUpdatedDt"),
    F.lit(executing_user).alias("SysUpdatedBy")
)

# ──────────────────────────────────────────────────────────────────────────────
# 4. DATA QUALITY & DEDUPLICATION
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Running Data Quality Checks...")

dq_df = silver_df.withColumn(
    "DQ_Reason",
    F.array_remove(F.array(
        F.when(F.col("WHSrcId") == "", F.lit("MISSING_WH_ID")),
        F.when(F.col("DivSrcId") == "", F.lit("MISSING_DIV_ID")),
        F.when(F.col("WHKey") == "", F.lit("MISSING_WH_KEY"))
    ), None)
).withColumn(
    "DQ_Status",
    F.when(F.size("DQ_Reason") > 0, F.lit(2)).otherwise(F.lit(0))
)

window_dup = Window.partitionBy("WHSrcId").orderBy(F.col("UpdatedDt").desc_nulls_last())

dq_df = dq_df.withColumn(
    "rn", F.row_number().over(window_dup)
).withColumn(
    "DQ_Status",
    F.when(F.col("rn") > 1, F.lit(4)).otherwise(F.col("DQ_Status"))
).drop("rn")

# Cache to prevent DAG explosion
dq_df.cache()

valid_df = dq_df.filter(F.col("DQ_Status") == 0).drop("DQ_Reason", "DQ_Status")
error_df = dq_df.filter(F.col("DQ_Status").isin(2, 4))

valid_count = valid_df.count()
error_count = error_df.count()

logger.info(f"✅ Valid Records: {valid_count:,}")
logger.info(f"❌ Error Records: {error_count:,}")

# ──────────────────────────────────────────────────────────────────────────────
# 5. CENTRAL EXCEPTION ROUTING
# ──────────────────────────────────────────────────────────────────────────────
if error_count > 0:
    logger.info("Routing Exceptions to Unity Catalog Quarantine...")
    exception_df = error_df.select(
        F.lit(SILVER_TABLE).alias("table_name"),
        F.col("WHSrcId").alias("record_key"),
        F.when(F.col("DQ_Status") == 4, F.lit("Duplicate Record"))
         .otherwise(F.concat(F.lit("Validation Failed: "), F.concat_ws(", ", F.col("DQ_Reason")))).alias("exception_details"),
        F.current_timestamp().alias("syscreateddt")
    )

    if spark.catalog.tableExists(EXCEPTION_TABLE):
        exception_df.write.format("delta").mode("append").saveAsTable(EXCEPTION_TABLE)
    else:
        exception_df.write.format("delta").mode("overwrite").option("path", EXCEPTION_PATH).saveAsTable(EXCEPTION_TABLE)

# ──────────────────────────────────────────────────────────────────────────────
# 6. MERGE INTO SILVER DELTA
# ──────────────────────────────────────────────────────────────────────────────
if valid_count > 0:
    logger.info("Executing MERGE INTO Silver Delta...")
    window_spec = Window.partitionBy(*MERGE_KEYS).orderBy(F.col("UpdatedDt").desc_nulls_last())
    final_df = valid_df.withColumn("rn", F.row_number().over(window_spec)).filter("rn = 1").drop("rn")

    if spark.catalog.tableExists(SILVER_TABLE):
        delta_tbl = DeltaTable.forName(spark, SILVER_TABLE)
        cond = " AND ".join([f"tgt.{k} = src.{k}" for k in MERGE_KEYS])
        
        (delta_tbl.alias("tgt")
         .merge(final_df.alias("src"), cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        final_df.write.format("delta").mode("overwrite").option("path", SILVER_PATH).saveAsTable(SILVER_TABLE)

dq_df.unpersist()
print("🎉 PROD WAREHOUSE PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM prx.prx_silver.warehouse LIMIT 10;

In [0]:
%sql
show tables in prx.prx_silver